In [1]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Annotated
from pydantic import BaseModel, Field
import operator
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
hf_token=os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")

In [3]:
llm1 = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="chat-completion",
    huggingfacehub_api_token=hf_token,
)

model1 = ChatHuggingFace(llm=llm1)


d:\LangGraph\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:



class Eassy_state(TypedDict):
    essay:str

    clearity_feedback:str
    language_feedback:str
    analysis_feedback:str
    overall_feedback:str

    individual_score:Annotated[list[int],operator.add]
    final_score:int
    

In [5]:
class Evolutionscema(BaseModel):
   Feedback:str=Field(description="provide the feedback ")
   score:int=Field(description="give the score between 1 to 10",ge=0 ,le=10)

parser=PydanticOutputParser(pydantic_object=Evolutionscema)


In [6]:
prompt1 = PromptTemplate(
    template="""
You are an evaluator.
Evaluate the essay below based on clarity of thought.

Essay:
{essay}

{format_instruction}
""",
    input_variables=["essay"],
    partial_variables={
        "format_instruction": parser.get_format_instructions()
    }
)
prompt2 = PromptTemplate(
    template="""
You are an evaluator.
Evaluate the essay below based on language.

Essay:
{essay}

{format_instruction}
""",
    input_variables=["essay"],
    partial_variables={
        "format_instruction": parser.get_format_instructions()
    }
)
prompt3 = PromptTemplate(
    template="""
You are an evaluator.
Evaluate the essay below based on depth of analysis.

Essay:
{essay}

{format_instruction}
""",
    input_variables=["essay"],
    partial_variables={
        "format_instruction": parser.get_format_instructions()
    }
)

prompt=PromptTemplate(template="summarise these feedback \n clearity of thought feedback {feedback1},\nlanguage feedback {feedback2},\ndepth of analysis feedback {feedback3}",
                      input_variables=["feedback1","feedback2","feedback3"])

In [7]:
def cot_feedback(state:Eassy_state):
    input=state["essay"]
    chain=prompt1|model1|parser
    result=chain.invoke({"essay":input})
    feedback=result.Feedback
    score=[result.score]

    return {"clearity_feedback":feedback,"individual_score":score}


In [8]:
def language_feedback(state:Eassy_state):
    input=state["essay"]
    chain=prompt2|model1|parser
    result=chain.invoke({"essay":input})
    feedback=result.Feedback
    score=[result.score]

    return {"language_feedback":feedback,"individual_score":score}

In [9]:
def dop_feedback(state:Eassy_state):
    input=state["essay"]
    chain=prompt3|model1|parser
    result=chain.invoke({"essay":input})
    feedback=result.Feedback
    score=[result.score]

    return {"analysis_feedback":feedback,"individual_score":score}

In [10]:
def final_feedback(state:Eassy_state):
    scores = state["individual_score"]
    final_score = sum(scores) // len(scores)
    feedback1=state["clearity_feedback"]
    feedback2=state["language_feedback"]
    feedback3=state["analysis_feedback"]
    chain=prompt|model1
    feedback=chain.invoke({"feedback1":feedback1,"feedback2":feedback2,"feedback3":feedback3}).content

    return {"overall_feedback":feedback,"final_score": final_score}

In [11]:
graph=StateGraph(Eassy_state)

graph.add_node("cot_feedback",cot_feedback)
graph.add_node("language_feedback",language_feedback)
graph.add_node("dop_feedback",dop_feedback)
graph.add_node("final_feedback",final_feedback)

graph.add_edge(START,"cot_feedback")
graph.add_edge(START,"language_feedback")
graph.add_edge(START,"dop_feedback")

graph.add_edge("cot_feedback","final_feedback")
graph.add_edge("language_feedback","final_feedback")
graph.add_edge("dop_feedback","final_feedback")

graph.add_edge("final_feedback",END)

In [ ]:
workflow=graph.compile()
workflow

In [ ]:
essay="""Artificial intelligence is transforming the way humans work, learn, and make decisions.
It enables machines to analyze large amounts of data and find patterns faster than humans.
AI is widely used in healthcare, education, finance, and transportation.
While it increases efficiency, it also raises ethical and privacy concerns.
Responsible development of AI is essential for a balanced and beneficial future"""
initial_state={
    "essay":essay}
workflow.invoke(initial_state)